# Data observability: Nordic Crown and Nordic Pearl AIS data

Measures how well the data meets our expectations (required columns, valid values, etc.).
The same checks run on the **raw** data and on the **cleaned** data, so we can show what cleaning fixed.

# Observability - IBM 

## Definition 

**Definition**: Ability to understand the internal state or condition, **e.g. how healthy the data is**, of a complex system based only on knowledge of its external outputs (specifically its telemetry = data a system sends out on its own, so it can be monitored from a distance).

We receive AIS data broadcast from Nordic Pearl and Crown (**telemetry**), and from those external outputs we can infer (work out) the internal state: **in port or at sea**, **operating mode**, etc.

## Why it matters 

Most organizations do not trust their data; they believe it is unreliable. Bad data can cause huge financial and hazardous problems for companies.
Traditionally, it has been difficult to identify bad data before it is too late, as the effect is not immediate (unlike when an IT server is down). It can take a long time to figure out that the data you are looking at is incorrect. 

**Data Observability**: monitors data pipelines to ensure complete, accurate, and timely delivery of data. 

**Data Pipeline**: Chain of steps the data goes through, from raw source to final result. 

    1. Source: AIS data collected.
    2. Load: CSV is read into pandas.
    3. Clean: Remove duplicates, invalid codes and outliers.
    4. Transform: Split the data into voyages and create features such as speed per leg, port time, etc.
    5. Analyse: EDA (Exploratory Data Analysis), plots and ML. 
    6. Output: Figures and results for the report.

**Exploratory Data Analysis**: An investigative process where we analyse and visualise a dataset to understand its structure, spot anomalies, check assumptions and uncover hidden patterns before building a statistical or ML model.

## 5 Pillars of data observability 

### 1. Freshness: 

Freshness describes how up to date your data is and how frequently it is updated. **Data staleness** occurs when there are important gaps in time when the data has not been updated. When data pipelines fail, it is often a freshness issue. Stale data is dangerous because it "fails silently": it does not crash anything, so the pipeline may keep running on old data.

### 2. Distribution: 

An indicator of the data's field-level health: distribution refers to whether or not the data falls within an accepted range. Deviation from the expected distribution may indicate data quality issues, errors or changes in the underlying data sources. **Speed range: 0–25 knots**

### 3. Volume:

Refers to the amount of data being generated, ingested, transformed and moved through various processes and pipelines. It also refers to the completeness of your data tables. It is a key **indicator of whether or not your data intake meets expected thresholds**. 

### 4. Schema: 

Schema describes the **organization of your data**. Schema changes often result in broken data. Data observability helps ensure that your data is organized consistently, is compatible across different systems and maintains its integrity throughout its lifecycle.

### 5. Lineage: 

Lineage is supposed to answer **"Where?"**, when the data breaks. It follows the data from its source to the end and notes changes, including what changed, why it changed and how it changed along the way. **Often represented visually**.

## Hierarchy

### 1. Dataset and pipeline level: Schema -> Volume (rows per ship, rows per day) -> Freshness (time range, gaps)
### 2. Column level: (nulls, data not available, ranges)
### 3. Row level: (duplicates, rows that break rules, etc.)




## Step 1: Load the raw data

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Load the file **as is**, with all columns and no cleaning. The timestamp is read as text and then parsed with `errors='coerce'`, so unparseable values become `NaT` and get counted instead of stopping the load.

Columns we do not need (for example `geom` and `geometry_wkt`, which repeat `lon` and `lat`) are kept here to show the data as received; they are dropped in the cleaning notebook.

In [ ]:
DATA_PATH = Path('..', 'data', 'ais_221026-ramleh.csv')

raw = pd.read_csv(DATA_PATH, low_memory=False)
raw['date_time_utc'] = pd.to_datetime(raw['date_time_utc'], errors='coerce')

print(f'{len(raw):,} rows, {raw.shape[1]} columns, '
      f'{raw.memory_usage(deep=True).sum() / 1e6:.0f} MB in memory')
raw.head()

In [ ]:
raw.info()

## Step 2: Are all the columns we need present?

**Pillar: Schema.** Before we can check anything else, the columns we rely on must exist in the data. If expected columns are missing, the other checks would crash; hence we begin with schema.

What we do:
1. **List the columns we need** (`required_columns`), with a short reason for each. We base the list on what the project needs and on the columns the cleaning notebook has rules for.
2. **Check that none are missing** (`missing_cols`): every required column that is not in `raw`. An empty list means all are present.
3. **Test the check** by running it on a copy without `lat`, to make sure it really finds a missing column.

In [ ]:
# Step 2: Columns we need for the project, and why
required_columns = [
    'date_time_utc',  # when the message was sent: needed for gaps and messages per day
    'mmsi',           # which ship: our ship ID 
    'lat',            # needed for the route and position analysis
    'lon',            # needed for the route and position analysis
    'speed',          # needed for energy consumption and speed analysis
    'cog',            # course over ground, needed for direction analysis
    'true_heading',   # needed for direction analysis
    'rot',            # rate of turn, needed for direction analysis
    'draught',        # needed for energy consumption and speed analysis    
]

missing_cols = [col for col in required_columns if col not in raw.columns ]
print('Missing columns:', missing_cols)


The code runs and shows that none of the required columns are missing. Note that we use `mmsi` as the ID, because the vessels were renamed in 2025.

In [ ]:
test = [col for col in required_columns if col not in raw.drop(columns=['lat']).columns]
print('Test, should find lat:', test)


Our test works, as it finds the missing column, which was dropped with `raw.drop()`.

## Step 3: Do the columns have the right data types?

**Pillar: Schema.** The columns must exist *and* hold the right kind of values: the timestamp must be a datetime, and the measurements must be numbers. Otherwise the checks in the next steps will fail or give wrong results.

What we do:
1. **Check each required column's type**: `date_time_utc` must be a datetime, and the rest must be numeric.
2. **Test the check** on a copy where `speed` is turned into text, to make sure it finds a wrong type.

In [ ]:
wrong_types = []
for col in required_columns:
    if col == 'date_time_utc':
        ok = pd.api.types.is_datetime64_any_dtype(raw[col])
    else:
        ok = pd.api.types.is_numeric_dtype(raw[col])
    if not ok:
        wrong_types.append(col)

if not wrong_types:
    print('Type check passed: all required columns have the expected type.')
else:
    print('Type check FAILED. Columns with wrong type:', wrong_types)


In [ ]:
# Test: break speed on purpose, then run the same check
test_df = raw[required_columns].copy()
test_df['speed'] = test_df['speed'].astype(str)

test_wrong_types = []
for col in required_columns:
    if col == 'date_time_utc':
        ok = pd.api.types.is_datetime64_any_dtype(test_df[col])
    else:
        ok = pd.api.types.is_numeric_dtype(test_df[col])
    if not ok:
        test_wrong_types.append(col)

if not test_wrong_types:
    print('Type check passed: all required columns have the expected type.')
else:
    print('Type check FAILED. Columns with wrong type:', test_wrong_types)



We created a copy of `raw` and changed the values of `speed` from numeric to text. The check works, as it returned the expected wrong type: `speed`.

## Step 4: How much data do we have per ship?

**Pillar: Volume.** We check that the amount of data is what we expect: both ships should be present, and each should have a reasonable number of messages.

What we do:
1. **Count rows per ship** (`mmsi`), to see that both ships are present and how much data each has.

**Expected:** 2 ships. AIS Class A ships report every 6 s at ferry speed (ITU-R M.1371),
so one year gives at most 365 × 86,400 / 6 ≈ 5.3 M rows per ship.


In [ ]:
# Step 4: Number of rows per ship
rows_per_ship = raw.groupby('mmsi').size()
print(rows_per_ship)
print(f'\nNumber of ships: {raw["mmsi"].nunique()}')


As expected, 2 ships. They have 1.69 M and 1.44 M rows, well below the 5.3 M maximum,
so there are gaps in the data.


### Messages per day

A total per ship can hide problems. We count messages per ship per day to see if any days are missing.

What we do:
1. **Count messages per ship per day.**
2. **Find days with no data** by comparing against every day in the period.
3. **Plot messages per day**, one panel per ship.


In [ ]:
# Messages per ship per day
raw['date'] = raw['date_time_utc'].dt.date
msgs_per_day = raw.groupby(['mmsi', 'date']).size()

all_days = pd.date_range('2025-01-01', '2025-12-31', freq='D').date


for mmsi in rows_per_ship.index:
    days_with_data = msgs_per_day.loc[mmsi].index
    missing_days = [d for d in all_days if d not in days_with_data]
    print(f'{mmsi}: {len(days_with_data)} of {len(all_days)} days have data, '
          f'median {msgs_per_day.loc[mmsi].median():.0f} messages per day')
    print('  Missing days:', [str(d) for d in missing_days])


In [ ]:
# Messages per day, one panel per ship, with missing days marked
colors = {219592000: '#2a78d6', 219945000: '#eb6834'}
names = {219592000: 'Nordic Crown', 219945000: 'Nordic Pearl'}

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, mmsi in zip(axes, rows_per_ship.index):
    daily = msgs_per_day.loc[mmsi].reindex(all_days, fill_value=0)  # missing days -> 0
    missing = daily[daily == 0].index

    ax.plot(pd.to_datetime(daily.index), daily.values, color=colors[mmsi], linewidth=1.5)
    for d in missing:
        ax.axvline(pd.to_datetime(d), color='#d03b3b', linewidth=1, linestyle='--')

    ax.set_title(f'{names[mmsi]} ({mmsi}), {len(missing)} missing days (red)', loc='left')
    ax.set_ylabel('Messages per day')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('AIS messages per day, 2025', x=0.01, ha='left')
plt.tight_layout()
plt.show()



Nordic Crown is missing 2 days and Nordic Pearl 3. 3 January is missing for
both, which points to a gap at the data source, and the data ends on 30 December,
so 31 December is missing for both. The daily count alternates between full and
nearly empty days because the receiver mainly covers the Oslo end of the route.
Both ships also have a period with very little data (Nordic Crown in March,
Nordic Pearl in January).


### Duplicate rows

**Expected:** no ship sends two messages with the same timestamp.


In [ ]:
# Rows where the same ship has the same timestamp more than once
duplicates = raw.duplicated(subset=['mmsi', 'date_time_utc'])
print('Duplicate rows:', duplicates.sum())


4,307 duplicate rows (about 0.1 % of the data), which should be removed in cleaning.

## Step 5: Are the values sensible?

**Pillar: Distribution.**

### Missing values

**Expected:** no missing values in the required columns.


In [ ]:
# Missing values per required column
missing_values = raw[required_columns].isna().sum()
print(missing_values)


### Valid ranges

**Expected:** all values are within the limits used in the cleaning notebook:

| Column | Valid | Why |
|---|---|---|
| lat | between 55 and 60 | the Oslo–Copenhagen route |
| lon | between 9 and 13 | the Oslo–Copenhagen route |
| speed | below 25 kn | the ferries sail at 16–18 kn |
| cog, true_heading | below 360° | 360 and 511 mean "not available" |
| rot | not −128 | −128 means "not available" |
| draught | below 8 m | 11.8 m was an error |


In [ ]:
# Number of rows outside the valid range, per column
invalid = {
    'lat':          (raw['lat'] <= 55) | (raw['lat'] >= 60),
    'lon':          (raw['lon'] <= 9) | (raw['lon'] >= 13),
    'speed':        raw['speed'] >= 25,
    'cog':          raw['cog'] >= 360,
    'true_heading': raw['true_heading'] >= 360,
    'rot':          raw['rot'] == -128,
    'draught':      raw['draught'] >= 8,
}

for col, bad in invalid.items():
    print(f'{col}: {bad.sum()} invalid rows')


6,409 rows are missing `rot` and `true_heading`, and a small number of rows
(2–84 per column) are outside the valid range. In total this is well under 1 %
of the data, and all of it is removed in cleaning.


## Step 6: Does the data cover the whole period?

**Pillar: Freshness.** The data is a single file for 2025, so instead of asking
"is new data arriving?" we check that it covers the whole year and has no long gaps.

### Time coverage

**Expected:** data from 1 January to 31 December 2025 for both ships.


In [ ]:
# First and last message per ship
coverage = raw.groupby('mmsi')['date_time_utc'].agg(['min', 'max'])
print(coverage)


Both ships start on 1 January and end on 30 December 2025. The data ends one day before the calendar year ends.

### Longest gap between messages

**Expected:** no gap longer than 24 hours. Shorter gaps are normal, because the
receiver does not cover the whole route, but a full day without messages means
data is missing.


In [ ]:
# Time between consecutive messages per ship
raw = raw.sort_values(['mmsi', 'date_time_utc'])
raw['gap'] = raw.groupby('mmsi')['date_time_utc'].diff()

for mmsi in rows_per_ship.index:
    ship = raw[raw['mmsi'] == mmsi]
    longest = ship['gap'].max()
    ends_at = ship.loc[ship['gap'].idxmax(), 'date_time_utc']
    print(f'{mmsi}: longest gap {longest}, ending at {ends_at}')


Both vessels have a gap of more than 24 hours (around 3 January), which is the same missing day found in the volume check.

## Step 7: What did cleaning fix?

**Pillar: Lineage.** We run the same checks on the raw data and on the cleaned
data from the cleaning notebook, and compare the results in one table.

**Expected:** cleaning removes duplicates, missing values and invalid values.
Missing days and gaps stay the same, since cleaning cannot add data that was never received
(this turned out not to hold, see the result below).


In [ ]:
CLEAN_PATH = Path('..', 'data', 'dataset_clean.csv')

clean = pd.read_csv(CLEAN_PATH, low_memory=False)
clean['date_time_utc'] = pd.to_datetime(clean['date_time_utc'], errors='coerce')

print(f'{len(clean):,} rows, {clean.shape[1]} columns')


In [ ]:
def run_checks(df):
    """Run all observability checks on a dataset and return the results."""
    df = df.sort_values(['mmsi', 'date_time_utc'])
    results = {}

    # Schema
    results['Missing required columns'] = len([c for c in required_columns if c not in df.columns])

    # Volume
    results['Rows'] = len(df)
    results['Ships'] = df['mmsi'].nunique()
    days_with_data = df.groupby('mmsi')['date_time_utc'].apply(lambda s: s.dt.date.nunique())
    results['Missing days (sum over ships)'] = int((len(all_days) - days_with_data).sum())
    results['Duplicate rows'] = int(df.duplicated(subset=['mmsi', 'date_time_utc']).sum())

    # Distribution
    results['Missing values'] = int(df[required_columns].isna().sum().sum())
    invalid = ((df['lat'] <= 55) | (df['lat'] >= 60)
               | (df['lon'] <= 9) | (df['lon'] >= 13)
               | (df['speed'] >= 25)
               | (df['cog'] >= 360)
               | (df['true_heading'] >= 360)
               | (df['rot'] == -128)
               | (df['draught'] >= 8))
    results['Rows outside valid range'] = int(invalid.sum())

    # Freshness
    results['First message'] = df['date_time_utc'].min()
    results['Last message'] = df['date_time_utc'].max()
    gap = df.groupby('mmsi')['date_time_utc'].diff()
    results['Longest gap (hours)'] = round(gap.max().total_seconds() / 3600, 1)

    return results


In [112]:
# What each check should give, written down before looking at the results
expected = {
    'Missing required columns':      0,
    'Rows':                          'at most 2 × 5.3 M',
    'Ships':                         2,
    'Missing days (sum over ships)': 0,
    'Duplicate rows':                0,
    'Missing values':                0,
    'Rows outside valid range':      0,
    'First message':                 '2025-01-01',
    'Last message':                  '2025-12-31',
    'Longest gap (hours)':           'at most 24',
}

summary = pd.DataFrame({
    'Expected': expected,
    'Raw': run_checks(raw),
    'Cleaned': run_checks(clean),
})
summary


,Expected,Raw,Cleaned
Missing required columns,0,0,0
Rows,at most 2 × 5.3 M,3134835,3126297
Ships,2,2,2
Missing days (sum over ships),0,5,78
Duplicate rows,0,4307,0
Missing values,0,12818,0
Rows outside valid range,0,158,0
First message,2025-01-01,2025-01-01 00:00:01,2025-01-01 00:00:01
Last message,2025-12-31,2025-12-30 23:02:05,2025-12-30 23:02:05
Longest gap (hours),at most 24,45.7,353.7


Cleaning removed all duplicates, missing values and invalid values, and only
0.3 % of the rows. However, missing days rose from 5 to 78 and the longest gap
from 46 h to 354 h. The 6,409 rows with missing `rot` and `true_heading` all
come from source S (probably satellite AIS), and on 73 days they were the only
data, so removing them emptied those days.

About 45 of these days are two yard stays, probably dry docking for a retrofit:
both ships lie still at the same position near Landskrona (55.86° N, 12.83° E),
Nordic Pearl from 2 to 27 January and Nordic Crown from 16 March to 4 April, and
both were renamed during the stay. The remaining days are sailing days with only
a few satellite messages. Since the ships were not in service during the yard
stays, losing these days has little effect on the analysis.

The renaming is consistent with the ships' renewal under their new owner
([Cruise & Ferry, "How Gotlandsbolaget is renewing its Nordic ferry experience"](https://www.cruiseandferry.net/articles/how-gotlandsbolaget-is-renewing-its-nordic-ferry-experience)).


## Conclusion

- **Schema:** all required columns are present and have the right types.
- **Volume:** both ships are present, with 1.69 M and 1.44 M rows. Coverage is mainly
  good near Oslo, so every second day has little data. 3 January and 31 December
  are missing, and there are 4,307 duplicate rows.
- **Distribution:** 6,409 rows lack `rot` and `true_heading`, and fewer than 200
  rows have invalid values.
- **Freshness:** the data covers 1 January to 30 December, with one gap over 24 hours
  (around 3 January).
- **Lineage:** cleaning removes all duplicates, missing values and invalid values
  (0.3 % of the rows), but it also removes 73 ship-days of satellite data, mostly
  two yard stays when the ships were out of service.

The data is fit for analysis, as long as we keep in mind that coverage is uneven
along the route. In a live pipeline these checks would run automatically on every
new delivery; here we run them once, on the raw and the cleaned data.
